# 品质因数
## 谐振电路和 Q 因数
谐振电路广泛用于振荡器、调谐放大器、滤波器等。在特定频率，即*谐振频率* $f_r$（或 $\omega_r$），谐振器呈现最大（或最小）阻抗（例如：开路或短路）[1]。品质因数 $Q$，或 Q 因数，是谐振无源电路损耗的无量纲优值系数，定义为 [2,3]：

$$
Q = 2 \pi \left. \frac{\textrm{平均储能}}{\textrm{每秒能量损耗}} \right|_{\omega=\omega_r}
=
\omega_r \left. \frac{\textrm{平均储能}}{\textrm{平均功率损耗}} \right|_{\omega=\omega_r}
$$

根据此定义，损耗越低意味着 $Q$ 越高。具有较高 Q 因数的谐振器在谐振频率处以更大的振幅谐振，但围绕该频率谐振的频率范围 $BW$ 较小。


## 有载和无载 Q 因数
实际上，根据考虑的损耗不同，可以定义三种 Q 因数 [2]：

* 无载 Q：$$Q_0 = \omega_r \left. \frac{\textrm{谐振电路中的储能}}{\textrm{谐振电路中的功率损耗}} \right|_{\omega=\omega_r}$$

* 外部 Q：$$Q_e = \omega_r \left. \frac{\textrm{谐振电路中的储能}}{\textrm{外部电路中的功率损耗}} \right|_{\omega=\omega_r}$$

* 有载 Q：$$Q_L = \omega_r \left. \frac{\textrm{谐振电路中的储能}}{\textrm{总功率损耗}} \right|_{\omega=\omega_r}$$

*有载* Q 因数 $Q_L$ 描述整个谐振系统中的能量耗散，包括谐振器本身和用于观察谐振的仪器 [3]。术语*加载*指的是外部电路对测量量的影响。外部电路的任何损耗机制都会降低 $Q$。

*无载* Q 因数 $Q_0$ 是谐振器本身的特性，在没有外部电路引起的任何加载效应的情况下（未耦合）。对于大多数应用，期望的量是无载 Q 因数 $Q_0$，它仅由与谐振器相关的能量耗散决定，因此能最好地描述谐振模式。由于测量系统的加载效应，通常无法直接测量谐振器的无载 Q。然而，可以从有载谐振器的频率响应测量中估计 $Q_0$。

外部电路中的能量耗散由*外部* Q 因数 $Q_e$ 表征，这三个 Q 因数通过以下关系式相关联（由上述三个表达式推导得出）：

$$
\frac{1}{Q_L} = \frac{1}{Q_e} + \frac{1}{Q_0} 
$$

如果定义*耦合系数* $\beta=Q_0/Q_e$，则：

$$
Q_0 = (1 + \beta) Q_L
$$

可以区分三种情况：

1. $\beta<1$：谐振器被称为与馈电电路*欠耦合*
2. $\beta=1$：谐振器被称为*临界耦合*
3. $\beta>1$：谐振器被称为*过耦合*


虽然有载品质因数 $Q_L$ 的测量很简单，但要获得低于 1% 的不确定度（这被认为是 Q 因数测量的低不确定度）需要注意实验程序的各个方面。此外，从测量的 S 参数中找到无载 $Q_0$ 包括首先找到耦合系数 $\beta$，然后从 3dB 带宽测量 $Q_L$ 并使用上述关系式。

幸运的是，`scikit-rf` 实现了从频域 S 参数自动确定*有载*和*无载* Q 因数的方法。实现的方法在 [4] 中有详细描述，可以应用于传输或反射测量。

## 并联 RLC 电路示例
为了说明 `scikit-rf` 的 `Qfactor` 类的用法，使用并联 RLC 电路作为示例，该电路具有可用于基准测试的解析公式。

<img src="figures/Parallel_RLC_resonator.svg" width="300"/>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

rf.stylely()

In [ ]:
C = 1e-6  # F
L = 1e-9  # H
R = 30  # Ohm
Z0 = 50  # Ohm

freq = rf.Frequency(5, 5.2, npoints=501, unit='MHz')
media = rf.media.DefinedGammaZ0(frequency=freq, z0=Z0)  # ideal line (no loss)
rng = np.random.default_rng()
random_d = rng.uniform(-np.pi, np.pi)  # a random length for the sake of the example

resonator = media.line(d=random_d, unit='rad') \
                **media.shunt_inductor(L) ** media.shunt_capacitor(C) \
                ** media.shunt(media.resistor(R)**media.short()) ** media.open()

resonator.plot_s_db()

此情况的解析公式可用，由 [2,3] 给出：
$$ f_{\mathrm{res}} = \frac{1}{2\pi \sqrt{L C}} $$
$$ Q = \omega_r R C = \frac{R C}{\sqrt{L C}}$$

In [ ]:
def f_res_RLC(L, C):
    return 1/(2*np.pi*np.sqrt(L*C))

def Q_RLC(R, L, C):
    return R * C / np.sqrt(L*C)

无载 Q 因数 $Q_0$ 是谐振器本身的特性，在没有外部电路引起的任何加载效应的情况下。当然，在实践中，谐振器连接到该电路，这会产生降低有载电路 Q 因数 $Q_L$ 的效果。

因此，当谐振器连接到传输线时，它会被端口阻抗 $Z_0$ 加载（假设传输线无损耗），该阻抗与固有谐振电阻并联连接。因此，等效谐振负载为 $R \parallel Z_0 = (R Z_0) / (R+Z_0)$：

In [ ]:
print(f'Theoretical Resonant Frequency: {f_res_RLC(L, C)/1e6} MHz')
print(f'Theoretical Loaded Q: Q_L = {Q_RLC((R*Z0)/(R+Z0), L, C)}')  # Req = R//Z0
print(f'Theoretical Unloaded Q: Q_0 = {Q_RLC(R, L, C)}')

首先，让我们创建一个 `Qfactor` 对象，传入谐振器 Network 和我们正在处理的谐振器类型：

In [ ]:
Q = rf.qfactor.Qfactor(resonator, res_type='reflection')

请注意，在多个谐振的情况下（[如此示例](../examples/qfactor/Finding_Dk_Df_from_resonance_fitting.ipynb)），建议还传入估计的谐振频率，以及估计的（数量级）Q 因数。

然后，我们使用 `fit` 方法来拟合有载 Q 因数 $Q_L$ 和谐振频率。返回的结果将有助于推导无载 Q 因数。

In [ ]:
res = Q.fit()
print(f'Fitted Resonant Frequency: f_L = {Q.f_L/1e6} MHz')
print(f'Fitted Loaded Q-factor: Q_L = {Q.Q_L}')

请注意，如果您打印 `Qfactor` 对象，也会显示这些结果：

In [ ]:
Q

最后，从拟合特性推导出无载 Q 因数：

In [ ]:
Q0 = Q.Q_unloaded(res)
print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

请注意，传入结果是可选的。调用 `.fit()` 方法将在内部存储优化结果，并在未传入特定优化结果时使用它：

In [ ]:
Q0 = Q.Q_unloaded()  # will use the latest optimized results performed with .fit()
print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

请注意，解析结果和拟合结果吻合良好，相对误差很小：

In [ ]:
print('Relative Error on Q_0:', (Q_RLC(R, L, C) - Q0)/Q_RLC(R, L, C))

还可以从拟合结果生成 Network。例如，我们可以将更多频率点的拟合谐振器模型与初始 RLC 谐振器进行比较：

In [ ]:
new_freq = rf.Frequency(5, 5.2, npoints=5001, unit='MHz')
fitted_network = Q.fitted_network(res, frequency=new_freq)

In [ ]:
resonator.plot_s_mag(label='Parallel RLC ', lw=2)
fitted_network.plot_s_mag(label='Fitted Model', lw=2, ls='--')

## Q 圆
在谐振附近，谐振器可以用等效电路模型表示 [1,4]。

包括谐振器的射频电路的散射参数（反射或传输）在复平面 $(\Re(s), \Im(s))$ 中具有圆的形式。S 参数可以表示为频率 $f$、有载品质因数 $Q_L$ 的函数 [1,4]：

$$
S(f) = S_D + d \frac{e^{-2 j \delta}}{1 + j Q_L \mathcal{F}}
$$

其中：
- $S_D$ 是在远高于或低于谐振的频率处测量的失谐 S 参数
- $Q_L$ 是有载 Q 因数
- $f_L$ 是有载谐振频率
- $f_0$ 是无载谐振频率
- $d$ 是 Q 圆的直径
- $\delta$ 是定义 Q 圆方向的实值常数
- $\mathcal{F}$ 是分数偏移频率，由下式给出：
$$
\mathcal{F} = \frac{f}{f_L} - \frac{f_L}{f}
$$

分数频率 $\mathcal{F}$ 是处理谐振电路时表达频率的便捷方式：在谐振频率处 $\mathcal{F}=0$，当 $f<f_L$ 时，$\mathcal{F}$ 为负，当 $f>f_L$ 时为正。如果 $\Delta f = f - f_L$ 是偏离谐振的频率偏差，则在谐振附近：

$$\mathcal{F} \approx 2 \frac{\Delta f}{f_L}$$

例如，如果源频率比谐振频率低 10%（$0.9 f/f_L$），则 $\mathcal{F}\approx -0.2$。

Q 圆的参数（直径以及调谐和失谐 S 参数）最终可以从以下方式获得：

In [ ]:
diam, S_V, S_T = Q.Q_circle()

使用极坐标平面检查结果：

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
resonator.plot_s_polar(ax=ax, label="RLC Resonator", ls='', marker='x', ms=5)
fitted_network.plot_s_polar(ax=ax, label="Fitted Model", lw=2)
ax.plot(np.angle(S_V), np.abs(S_V), 'ko')
ax.plot(np.angle(S_T), np.abs(S_T), 'ko')
ax.text(np.angle(S_T), 0.8*np.abs(S_T), '$S_T$')
ax.text(np.angle(S_V), 1.1*np.abs(S_V), '$S_V$')

## 带宽
Q 因数的另一个定义是传输谐振器的频率与带宽之比 [6]：

$$
Q_L = \frac{\omega_r}{\Delta \omega} = \frac{f_r}{\Delta f} 
$$

其中 $f_r$ 是谐振频率，$\Delta f=BW$ 是谐振宽度或*分数带宽* BW。请注意，此定义仅是近似值，当存在显著泄漏时可能不准确（有关更多信息，请参阅 [4]）。带宽的定义还取决于谐振器类型：

* 传输：带宽，也称为*半高全宽*（FWHM），是耗散功率为谐振频率处最大值一半（低于最大值 3 dB）的带宽。
* 反射：虽然可以将带宽定义为低于最大反射系数的 3 dB 点 [6]，但更适当的带宽定义包括测量频率 $f_1$ 和 $f_2$ 之间的差值，它们属于从 $S_T$ 到原点的中心线两侧倾斜 45 度的两个点（有关更多详细信息，请参阅 [4,8]）。

使用上述关系，带宽 BW 也可以通过 `Qfactor` 类的 `.BW` 参数获得。

In [ ]:
BW = Q.BW
print(f'Bandwidth: {BW} Hz')

In [ ]:
fig, ax = plt.subplots()
resonator.plot_s_db(label='Parallel RLC ', lw=2, ax=ax)
ax.axvspan(xmin=Q.f_L-Q.BW/2, xmax=Q.f_L+Q.BW/2, alpha=0.3, label='Bandwidth')
ax.legend()

## 参考文献
- [1] B. A. Galwas, "Scattering Matrix Description of Microwave Resonators," in IEEE Transactions on Microwave Theory and Techniques, vol. 31, no. 8, pp. 669-671, Aug. 1983, doi: 10.1109/TMTT.1983.1131566.
- [2] Peter A. Rizzi, "Microwave Engineering: Passive Circuits", Prentice-Hall, 1988 
- [3] David M. Pozar, "Microwave Engineering", 4th Edition, Éditeur	Wiley, 2011
- [4] A. P. Gregory, "Q-factor Measurement by using a Vector Network Analyser", National Physical Laboratory Report MAT 58 (2021), https://eprintspublications.npl.co.uk/9304/
- [5] Microwaves101, "Resonance of RLC Circuits", https://www.microwaves101.com/encyclopedias/resonance-of-rlc-circuits
- [6] Green, Estill I. (October 1955). "The Story of Q", American Scientist. 43: 584–594. http://www.collinsaudio.com/Prosound_Workshop/The_story_of_Q.pdf
- [7] R. S. Kwok and J.-F. Liang, 'Characterization of high-Q resonators for microwave filter applications', IEEE Transactions on Microwave Theory and Techniques, vol. 47, no. 1, pp. 111–114, Jan. 1999, doi: 10.1109/22.740093.
- [8] Darko Kajfez, "Q factor measurements, analog and digital", https://people.engineering.olemiss.edu/darko-kajfez/assets/rfqmeas2b.pdf